# Genotype comparison 

https://support.bioconductor.org/p/p133524/

In [ ]:
library_load <- suppressMessages(
    
    suppressWarnings(
    
            list(
                
                # Seurat 
                library(Seurat), 
                
                # RNA seq
                library(edgeR), 

                # Data 
                library(tidyverse), 
                library(openxlsx), 

                # Plotting 
                library(ComplexHeatmap), 
                library(circlize), 
                library(viridis), 
                library(ggplotify), 
                library(patchwork), 
                library(RColorBrewer), 

                # Python 
                library(reticulate)
            
            )
    )
)

In [ ]:
# Configure reticulate 
use_condaenv(condaenv='p.3.10.16-FD20250213HSCT', conda="/nobackup/peer/fdeckert/miniconda3/bin/conda", required=NULL)
py_config()

In [ ]:
options(warn=-1)
options(dplyr.summarise.inform=FALSE)

In [ ]:
random_seed <- 42
set.seed(random_seed)

In [ ]:
# Set working directory to project root
setwd("/research/peer/fdeckert/FD20250213HSCT")

In [ ]:
# Source files
source("plotting_global.R")
source("bin/seurat_qc.R")
source("script/2_analysis/2_dea/pseudobulk/genotype/celltype_low/pl.R")

In [ ]:
# Plotting Theme
ggplot2::theme_set(theme_global_set(size_select=1)) # From project global source()

In [ ]:
grouping_var="genotype_class"

random_effect_var="patient_id"
random_effect_var_cell_min=1
random_effect_var_min=3

cnt_min=3
cell_min=1
feature_group_min=3

p_adj_thr=0.1
sample_weights=FALSE

# Helper function

In [ ]:
group_select <- function(so, grouping_var, random_effect_var, random_effect_var_cell_min, random_effect_var_min) {
    
    # Get cells grouping information 
    cells_grouping <- so@meta.data[, c(grouping_var, random_effect_var)]
    
    # Count cells per group 
    cells_grouping_n <- cells_grouping %>% 
    
        # Count and filter random_effect_var by random_effect_var_cell_min
        dplyr::group_by_all() %>% 
        dplyr::summarise(random_effect_var_cell_n=n()) %>% 
        dplyr::mutate(random_effect_var_cell_check=ifelse(random_effect_var_cell_n >= random_effect_var_cell_min, TRUE, FALSE)) %>% 
        dplyr::filter(random_effect_var_cell_check) %>% 
    
        # Count and filter grouping_var by random_effect_var_min
        dplyr::group_by(across(grouping_var)) %>% 
        dplyr::mutate(random_effect_var_n=sum(random_effect_var_cell_check)) %>% 
        dplyr::mutate(random_effect_var_check=ifelse(random_effect_var_n >= random_effect_var_min, TRUE, FALSE)) %>% 
        dplyr::filter(random_effect_var_check) %>% 

        # Ungroup 
        dplyr::ungroup() %>% as.data.frame()
    
    # Set cell id filter
    cells_logical <- paste(so@meta.data[[random_effect_var]], so@meta.data[[grouping_var]]) %in% paste(cells_grouping_n[[random_effect_var]], cells_grouping_n[[grouping_var]])
    
    # Set group filter 
    group_logical <- length(unique(cells_grouping_n[[grouping_var]])) > 1
    
    # Filter by cell id and return Seurat object if any cells are found
    if(any(cells_logical) & group_logical) {
        
        so <- subset(so, cells=colnames(so)[cells_logical])
        
    } else {
        
        so <- NULL
    
    }
    
    return(so)
    
}

In [ ]:
feature_select <- function(so, cnt_min, cell_min, grouping_var, random_effect_var, feature_group_min, verbose=TRUE) {
    
    # Select Seurat components 
    meta <- so@meta.data
    cnt <- GetAssayData(so, assay="RNA", slot="counts")
    
    # Prepare data for split 
    cnt <- t(as.matrix(cnt))
    cnt <- as.data.frame(cnt)
    
    # Split expression matrix by grouping variable into list
    cnt <- split(cnt, f=paste(so[[grouping_var, drop=TRUE]], so[[random_effect_var, drop=TRUE]]))
    
    # Check within group expression 
    cnt <- lapply(cnt, function(x) {colSums(x>=cnt_min)>=cell_min})
    cnt <- do.call(rbind, cnt)
    
    # Check across group expression 
    cnt_check <- colSums(cnt)>=feature_group_min
    
    # Get genes 
    genes <- colnames(cnt)[cnt_check]

    # Create Seurat object with gene subset 
    cnt <- GetAssayData(so, assay="RNA", slot="counts")
    cnt <- cnt[genes, , drop=FALSE]

    so <- CreateSeuratObject(counts=cnt, meta=meta)
    
    if(verbose) {message(paste("Final number of genes for testing", nrow(so)))}

    return(so)
    
}

In [ ]:
voom_lm_fit <- function(so, grouping_var, random_effect_var, sample_weights) {

    # Get Counts 
    cnt <- GetAssayData(so, assay="RNA", slot="counts")
        
    # Prepare count data for split 
    cnt <- t(as.matrix(cnt))
    cnt <- as.data.frame(cnt)
        
    # Make pseudobulks by suming single cell counts
    cnt <- split(cnt, f=paste0(so[[grouping_var, drop=TRUE]], "|", so[[random_effect_var, drop=TRUE]]))
    cnt <- lapply(names(cnt), function(i) {x <- data.frame(counts=colSums(cnt[[i]])); colnames(x) <- i; return(x)})
    cnt <- do.call(cbind, cnt)
        
    # Get grouping variables from cnt matrix
    grouping_var_vec <- sapply(strsplit(colnames(cnt), "\\|"), `[[`, 1) # Used for design matrix
    random_effect_var_vec <- sapply(strsplit(colnames(cnt), "\\|"), `[[`, 2) # Used as blocking variable
    
    # Design matrix 
    grouping_var_vec <- as.character(grouping_var_vec)
    design <- model.matrix(~0+grouping_var_vec)
    colnames(design) <- gsub("grouping_var_vec", "", colnames(design))
    
    # Run voomLmFit
    fit <- edgeR::voomLmFit(
        
        counts=cnt, 
        design=design, 
        # block=random_effect_var_vec, 
        normalize.method="none", 
        lib.size=NULL, 
        sample.weights=sample_weights, 
        plot=FALSE

    )
    
    return(fit)
    
}

In [ ]:
ebayes <- function(fit, so) {
    
    # Get counts for percentage expression 
    cnt <- GetAssayData(so, assay="RNA", slot="counts")
    
    # Get grouping var and design from fit
    grouping_var_vec <- colnames(fit$design)
    
    # Define result list
    result <- list()
    
    # Set groups for contrast
    grouping_var_vec <- grouping_var_vec[grouping_var_vec %in% c("Host", "Donor")]

    # Set contrast 
    contrast_df <- data.frame(

        contrast_1 = c("Donor"), 
        contrast_2 = c("Host")

    )

    # Filter contrasts
    contrast_df <- contrast_df[contrast_df$contrast_1 %in% grouping_var_vec & contrast_df$contrast_2 %in% grouping_var_vec, , drop=FALSE]
    
    if(!nrow(contrast_df > 0)) {
        
        message("No contrasts")
        
    } else {
        
        for(i in 1:nrow(contrast_df)) {

            # Contrast fit
            contrasts_vec <- paste0(contrast_df$contrast_1[i], "-", contrast_df$contrast_2[i])
            contrasts <- limma::makeContrasts(contrasts=contrasts_vec, levels=grouping_var_vec)
            contrasts_fit <- limma::contrasts.fit(fit, contrasts=contrasts)

            # eBayes fit 
            efit <- limma::eBayes(contrasts_fit)

            # Get result table
            result_df <- limma::topTable(efit, sort.by="P", n=Inf, p.value=1, lfc=0, coef=1)

            # Convert results to Seurat format 
            colnames(result_df)[1] <- "avg_log2FC"
            colnames(result_df)[4] <- "p_value"
            colnames(result_df)[5] <- "p_val_adj"

            # Adjusted p-values with IHW
            result_df$p_val_adj <- IHW::adj_pvalues(IHW::ihw(result_df$p_value ~ result_df$AveExpr, alpha=0.05))
            
            # Add percentage expression 
            cnt_bool <- cnt[rownames(result_df), ] > 0
            
            result_df$pct_1 <- rowSums(cnt_bool[, so[[grouping_var, drop=TRUE]]==contrast_df$contrast_2[i]])/sum(so[[grouping_var, drop=TRUE]]==contrast_df$contrast_2[i])
            result_df$pct_2 <- rowSums(cnt_bool[, so[[grouping_var, drop=TRUE]]==contrast_df$contrast_1[i]])/sum(so[[grouping_var, drop=TRUE]]==contrast_df$contrast_1[i])

            result[[contrasts_vec]] <- result_df

        }
        
    }

    return(result)
    
}

# Run DEA for Skin HSCT

In [ ]:
so <- readRDS("data/object/pp.rds")

In [ ]:
# Subset data by hto demux class
so <- subset(so, subset=hto_demux_class=="Skin")

In [ ]:
# Set time points 
so$time_point <- ifelse(so$time_point %in% c("D14", "D100"), "HSCT", as.character(so$time_point))
so$time_point <- factor(so$time_point, levels=c("Baseline", "Tx", "HSCT", "GVHD"))

In [ ]:
# Subset for time point
so <- subset(so, subset=time_point=="HSCT")

In [ ]:
# Subset by celltype domain 
so <- subset(so, subset=celltype_domain=="Immune~cells")

In [ ]:
# Split patient by genotype
so$patient_id <- paste0(so$patient_id, "_", so$genotype_class)

In [ ]:
# Drop levels 
so@meta.data <- so@meta.data %>% droplevels()

In [ ]:
# Prepare data 
so <- SplitObject(so, split.by="celltype_low")

In [ ]:
# Subset samples 
so <- lapply(so, group_select, grouping_var=grouping_var, random_effect_var=random_effect_var, random_effect_var_cell_min=random_effect_var_cell_min, random_effect_var_min=random_effect_var_min)
so <- so[!sapply(so, is.null)] # Remove empty cell types

In [ ]:
# Subset features
so <- lapply(so, feature_select, cnt_min=cnt_min, cell_min=cell_min, grouping_var=grouping_var, random_effect_var=random_effect_var, feature_group_min=feature_group_min)

In [ ]:
# DEA voom fit 
fit <- lapply(so, voom_lm_fit, grouping_var=grouping_var, random_effect_var=random_effect_var, sample_weights=sample_weights)

In [ ]:
# Get results 
results <- lapply(names(fit), function(i) ebayes(fit=fit[[i]], so=so[[i]]))
names(results) <- names(fit)

In [ ]:
# Order results by cell types 
cell_type_order <- names(color$celltype_low_r)
cell_type_order <- cell_type_order[cell_type_order %in% names(results)]

results <- results[cell_type_order]

In [ ]:
# Save results 
saveRDS(fit, "result/dea/pseudobulk/genotype/celltype_low/fit_skin_hsct.rds")
saveRDS(results, "result/dea/pseudobulk/genotype/celltype_low/results_skin_hsct.rds")
results_xlsx(results, "result/dea/pseudobulk/genotype/celltype_low", "_skin_hsct.xlsx")

## Heatmap summary

In [ ]:
# Grouping statistics
grouping_stat <- grouping_stat_df(so, grouping_var=grouping_var, random_effect_var=random_effect_var, file="result/dea/pseudobulk/genotype/celltype_low/grouping_stat_skin_hsct")

In [ ]:
# Heatmap summary 

hm_1 <- grouping_stat_hm(grouping_stat, stat="cell_mean", column_title="Cell count mean", name="log2(mean cells)", row_order=cell_type_order)
hm_2 <- grouping_stat_hm(grouping_stat, stat="random_effect_var_n", column_title="Patient ID count", name="Count (Patient ID)", row_order=cell_type_order)
hm_3 <- results_n_hm(results, p_adj_thr=p_adj_thr, column_title="DEG count", name="Count (DEG)", row_order=cell_type_order)

In [ ]:
options(repr.plot.width=15, repr.plot.height=12)

draw(hm_1 + hm_2 + hm_3, ht_gap=unit(2, "cm"))

## Volcano plots

In [ ]:
options(repr.plot.width=4*8, repr.plot.height=8)

vp_run(results, p_adj_thr=p_adj_thr)

# Run DEA for Skin GVHD

In [ ]:
so <- readRDS("data/object/pp.rds")

In [ ]:
# Subset data by hto demux class
so <- subset(so, subset=hto_demux_class=="Skin")

In [ ]:
# Set time points 
so$time_point <- ifelse(so$time_point %in% c("Tx", "D14", "D100"), "HSCT", as.character(so$time_point))
so$time_point <- factor(so$time_point, levels=c("Baseline", "HSCT", "GVHD"))

In [ ]:
# Subset for time point
so <- subset(so, subset=time_point=="GVHD")

In [ ]:
# Subset by celltype domain 
so <- subset(so, subset=celltype_domain=="Immune~cells")

In [ ]:
# Split patient by genotype
so$patient_id <- paste0(so$patient_id, "_", so$genotype_class)

In [ ]:
# Drop levels 
so@meta.data <- so@meta.data %>% droplevels()

In [ ]:
# Prepare data 
so <- SplitObject(so, split.by="celltype_low")

In [ ]:
# Subset samples 
so <- lapply(so, group_select, grouping_var=grouping_var, random_effect_var=random_effect_var, random_effect_var_cell_min=random_effect_var_cell_min, random_effect_var_min=random_effect_var_min)
so <- so[!sapply(so, is.null)] # Remove empty cell types

In [ ]:
# Subset features
so <- lapply(so, feature_select, cnt_min=cnt_min, cell_min=cell_min, grouping_var=grouping_var, random_effect_var=random_effect_var, feature_group_min=feature_group_min)

In [ ]:
# DEA voom fit 
fit <- lapply(so, voom_lm_fit, grouping_var=grouping_var, random_effect_var=random_effect_var, sample_weights=sample_weights)

In [ ]:
# Get results 
results <- lapply(names(fit), function(i) ebayes(fit=fit[[i]], so=so[[i]]))
names(results) <- names(fit)

In [ ]:
# Order results by cell types 
cell_type_order <- names(color$celltype_low_r)
cell_type_order <- cell_type_order[cell_type_order %in% names(results)]

results <- results[cell_type_order]

In [ ]:
# Save results 
saveRDS(fit, "result/dea/pseudobulk/genotype/celltype_low/fit_skin_gvhd.rds")
saveRDS(results, "result/dea/pseudobulk/genotype/celltype_low/results_skin_gvhd.rds")
results_xlsx(results, "result/dea/pseudobulk/genotype/celltype_low", "_skin_gvhd.xlsx")

## Heatmap summary

In [ ]:
# Grouping statistics
grouping_stat <- grouping_stat_df(so, grouping_var=grouping_var, random_effect_var=random_effect_var, file="result/dea/pseudobulk/genotype/celltype_low/grouping_stat_skin_gvhd")

In [ ]:
# Heatmap summary 

hm_1 <- grouping_stat_hm(grouping_stat, stat="cell_mean", column_title="Cell count mean", name="log2(mean cells)", row_order=cell_type_order)
hm_2 <- grouping_stat_hm(grouping_stat, stat="random_effect_var_n", column_title="Patient ID count", name="Count (Patient ID)", row_order=cell_type_order)
hm_3 <- results_n_hm(results, p_adj_thr=p_adj_thr, column_title="DEG count", name="Count (DEG)", row_order=cell_type_order)

In [ ]:
options(repr.plot.width=15, repr.plot.height=12)

draw(hm_1 + hm_2 + hm_3, ht_gap=unit(2, "cm"))

## Volcano plots

In [ ]:
options(repr.plot.width=4*8, repr.plot.height=8)

vp_run(results, p_adj_thr=p_adj_thr)

# Session info

In [ ]:
sessionInfo()